# nb

> Core notebook primitives for cell iteration, source extraction, and analysis

In [ ]:
#| default_exp utils.nb

In [ ]:
#| hide
from nbdev.showdoc import *

## Imports

In [ ]:
#| export
from __future__ import annotations
from typing import Any, Dict, List, Optional, Iterable, Tuple, Set, Callable

from dataclasses import dataclass

import ast
from pathlib import Path

from nbdev_mcp.utils.re import (
    EXPORT_ANY_RE,
    EXPORT_DIRECTIVE_PATTERN,
    EXPORTA_DIRECTIVE_PATTERN,
    DEFAULT_EXP_PATTERN,
    HIDE_DIRECTIVE_PATTERN,
    EVAL_FALSE_PATTERN,
)

## Source Handling

Functions for joining and processing cell source code. Defined first as `NbFile` depends on `join_source`.

In [ ]:
#| export
def join_source(source: str | List[str]) -> str:
    """Join notebook cell source lines into a single string.
    
    Parameters
    ----------
    source : str or List[str]
        Cell source as a string or list of strings (Jupyter format).
    
    Returns
    -------
    str
        Joined source code with proper newline handling.
    """
    if isinstance(source, str):
        return source
    if not source:
        return ''
    result = []
    for line in source:
        result.append(line if line.endswith('\n') else line + '\n')
    joined = ''.join(result)
    return joined.rstrip('\n') if joined.endswith('\n\n') else joined

In [ ]:
#| export
def truncate(s: str, limit: int = 1000, suffix: str = '...') -> str:
    """Truncate string to limit, adding suffix if truncated.
    
    Parameters
    ----------
    s : str
        String to truncate.
    limit : int
        Maximum length before truncation (default 1000).
    suffix : str
        Suffix to append when truncated (default '...').
    
    Returns
    -------
    str
        Original string if under limit, otherwise truncated with suffix.
    """
    if not s or len(s) <= limit:
        return s or ''
    return s[:limit] + f'\n{suffix} [truncated {len(s) - limit} chars]'

## Cell Data Classes

Simple data containers for notebook cells.

In [ ]:
#| export
@dataclass
class Cell:
    """A single notebook cell with its metadata.
    
    Parameters
    ----------
    index : int
        Zero-based cell index in the notebook.
    cell_type : str
        Cell type ('code' or 'markdown').
    source : str
        Cell source code as a single string.
    raw : Dict[str, Any]
        Original cell dictionary from notebook JSON.
    """
    index: int
    cell_type: str
    source: str
    raw: Dict[str, Any]
    
    @property
    def lines(self) -> List[str]:
        """Split source into lines."""
        return self.source.splitlines()
    
    @property
    def is_code(self) -> bool:
        """Check if this is a code cell."""
        return self.cell_type == 'code'
    
    @property
    def is_markdown(self) -> bool:
        """Check if this is a markdown cell."""
        return self.cell_type == 'markdown'

In [ ]:
#| export
@dataclass
class NBFile:
    """A notebook file with its path and data.
    
    Parameters
    ----------
    path : Path
        Path to the notebook file.
    data : Dict[str, Any]
        Parsed notebook JSON data.
    """
    path: Path
    data: Dict[str, Any]
    
    @property
    def cells(self) -> List[Cell]:
        """Get all cells as Cell objects."""
        return [
            Cell(
                index=i,
                cell_type=c.get('cell_type', 'code'),
                source=join_source(c.get('source', [])),
                raw=c
            )
            for i, c in enumerate(self.data.get('cells', []))
        ]
    
    def code_cells(self) -> Iterable[Cell]:
        """Iterate over code cells only."""
        return (c for c in self.cells if c.is_code)
    
    def markdown_cells(self) -> Iterable[Cell]:
        """Iterate over markdown cells only."""
        return (c for c in self.cells if c.is_markdown)

## Directive Detection

Functions for detecting nbdev directives in cell source.

In [ ]:
#| export
def has_export(source: str) -> bool:
    """Check if source has an export directive.
    
    Parameters
    ----------
    source : str
        Cell source code.
    
    Returns
    -------
    bool
        True if source contains ``#| export``, ``#| exporti``, or ``#| exporta``.
    """
    return bool(EXPORT_ANY_RE.search(source))

In [ ]:
#| export
def has_hide(source: str) -> bool:
    """Check if source has a hide directive.
    
    Parameters
    ----------
    source : str
        Cell source code.
    
    Returns
    -------
    bool
        True if source contains ``#| hide``.
    """
    return bool(HIDE_DIRECTIVE_PATTERN.search(source))

In [ ]:
#| export
def has_eval_false(source: str) -> bool:
    """Check if source has eval: false directive.
    
    Parameters
    ----------
    source : str
        Cell source code.
    
    Returns
    -------
    bool
        True if source contains ``#| eval: false``.
    """
    return bool(EVAL_FALSE_PATTERN.search(source))

In [ ]:
#| export
def get_default_exp(source: str) -> Optional[str]:
    """Extract the default_exp module name from source.
    
    Parameters
    ----------
    source : str
        Cell source code.
    
    Returns
    -------
    str or None
        Module name from ``#| default_exp <module>`` or None if not found.
    """
    m = DEFAULT_EXP_PATTERN.search(source)
    return m.group(1) if m else None

In [ ]:
#| export
def find_default_exp(nb: NBFile | Dict[str, Any]) -> Optional[str]:
    """Find the default_exp module name in a notebook.
    
    Parameters
    ----------
    nb : NBFile or Dict
        Notebook object or parsed notebook data.
    
    Returns
    -------
    str or None
        Module name from the first ``#| default_exp`` directive found.
    """
    cells = nb.cells if isinstance(nb, NBFile) else nb.get('cells', [])
    for cell in cells:
        if isinstance(cell, Cell):
            if cell.is_code:
                if exp := get_default_exp(cell.source):
                    return exp
        else:
            if cell.get('cell_type') == 'code':
                src = join_source(cell.get('source', []))
                if exp := get_default_exp(src):
                    return exp
    return None

## Symbol Extraction

Extract function/class definitions from source code.

In [ ]:
#| export
@dataclass
class Symbol:
    """A symbol (function, class, or variable) defined in code.
    
    Parameters
    ----------
    name : str
        Symbol name.
    kind : str
        Symbol type: 'function', 'class', or 'variable'.
    lineno : int
        Line number where defined (1-based, default 0 = unknown).
    """
    name: str
    kind: str
    lineno: int = 0

In [ ]:
#| export
def extract_symbols(source: str) -> List[Symbol]:
    """Extract top-level symbols from Python source.
    
    Parameters
    ----------
    source : str
        Python source code.
    
    Returns
    -------
    List[Symbol]
        List of symbols found at module level.
    """
    try:
        tree = ast.parse(source)
    except SyntaxError:
        return []
    
    symbols = []
    for node in ast.iter_child_nodes(tree):
        if isinstance(node, ast.FunctionDef):
            symbols.append(Symbol(node.name, 'function', node.lineno))
        elif isinstance(node, ast.AsyncFunctionDef):
            symbols.append(Symbol(node.name, 'function', node.lineno))
        elif isinstance(node, ast.ClassDef):
            symbols.append(Symbol(node.name, 'class', node.lineno))
        elif isinstance(node, ast.Assign):
            for target in node.targets:
                if isinstance(target, ast.Name):
                    symbols.append(Symbol(target.id, 'variable', node.lineno))
    return symbols

## Import Extraction

Extract import statements from source code.

In [ ]:
#| export
@dataclass
class Import:
    """An import statement.
    
    Parameters
    ----------
    module : str
        Module name being imported.
    is_relative : bool
        True if this is a relative import (default False).
    level : int
        Number of dots for relative imports (default 0).
    """
    module: str
    is_relative: bool = False
    level: int = 0

In [ ]:
#| export
def extract_imports(source: str) -> List[Import]:
    """Extract import statements from Python source.
    
    Parameters
    ----------
    source : str
        Python source code.
    
    Returns
    -------
    List[Import]
        List of imports found in the source.
    """
    try:
        tree = ast.parse(source)
    except SyntaxError:
        return []
    
    imports = []
    for node in ast.walk(tree):
        if isinstance(node, ast.Import):
            for alias in node.names:
                imports.append(Import(alias.name, is_relative=False))
        elif isinstance(node, ast.ImportFrom):
            module = node.module or ''
            is_rel = node.level > 0
            imports.append(Import(module, is_relative=is_rel, level=node.level))
    return imports

In [ ]:
#| export
def split_imports(imports: List[Import], lib_name: str) -> Tuple[Set[str], Set[str]]:
    """Split imports into internal and external.
    
    Parameters
    ----------
    imports : List[Import]
        List of imports to classify.
    lib_name : str
        Library name to match for internal imports.
    
    Returns
    -------
    Tuple[Set[str], Set[str]]
        (internal_modules, external_modules)
    """
    internal, external = set(), set()
    for imp in imports:
        if imp.is_relative:
            internal.add(imp.module or '.')
        elif imp.module.startswith(lib_name + '.') or imp.module == lib_name:
            internal.add(imp.module)
        else:
            top = imp.module.split('.')[0]
            external.add(top)
    return internal, external

## Iteration Helpers

High-level iteration over notebooks and cells.

In [ ]:
#| export
def iter_export_cells(nb: NBFile | Dict[str, Any]) -> Iterable[Cell]:
    """Iterate over cells that have export directives.
    
    Parameters
    ----------
    nb : NBFile or Dict
        Notebook object or parsed notebook data.
    
    Yields
    ------
    Cell
        Code cells containing export directives.
    """
    cells = nb.cells if isinstance(nb, NBFile) else [
        Cell(i, c.get('cell_type', 'code'), join_source(c.get('source', [])), c)
        for i, c in enumerate(nb.get('cells', []))
    ]
    for cell in cells:
        if cell.is_code and has_export(cell.source):
            yield cell

In [ ]:
#| export
def search_cells(
    nb: NBFile | Dict[str, Any],
    query: str,
    max_results: int = 10
) -> List[Tuple[int, Cell]]:
    """Search notebook cells for a query string.
    
    Parameters
    ----------
    nb : NBFile or Dict
        Notebook to search.
    query : str
        Case-insensitive search string.
    max_results : int
        Maximum number of results to return (default 10).
    
    Returns
    -------
    List[Tuple[int, Cell]]
        List of (cell_index, cell) tuples matching the query.
    """
    cells = nb.cells if isinstance(nb, NBFile) else [
        Cell(i, c.get('cell_type', 'code'), join_source(c.get('source', [])), c)
        for i, c in enumerate(nb.get('cells', []))
    ]
    q = query.lower()
    results = []
    for cell in cells:
        if q in cell.source.lower():
            results.append((cell.index, cell))
            if len(results) >= max_results:
                break
    return results

In [ ]:
#| export
def iter_project_cells(
    project: 'Path',
    cell_type: Optional[str] = 'code',
    with_source: bool = True
) -> Iterable[Tuple['Path', int, Cell]]:
    """Iterate over all cells in all notebooks in a project.
    
    This is a convenience function for tools that need to scan
    all cells across a project (e.g., linting, analysis).
    
    Parameters
    ----------
    project : Path
        Path to the nbdev project root.
    cell_type : str or None
        Filter by cell type ('code', 'markdown'), or None for all.
    with_source : bool
        If True, include source in Cell objects (default True).
    
    Yields
    ------
    Tuple[Path, int, Cell]
        (notebook_path, cell_index, Cell) tuples.
    
    Examples
    --------
    >>> for nb_path, idx, cell in iter_project_cells(project, 'code'):
    ...     if 'TODO' in cell.source:
    ...         print(f'{nb_path}:cell[{idx}]')
    """
    # Import here to avoid circular dependency
    from nbdev_mcp.utils.paths import iter_notebooks, read_nb
    
    for nb_path in iter_notebooks(project):
        data = read_nb(nb_path)
        nb = NBFile(nb_path, data)
        for cell in nb.cells:
            if cell_type and cell.cell_type != cell_type:
                continue
            yield nb_path, cell.index, cell

## Export

In [ ]:
#| hide
import nbdev; nbdev.nbdev_export()